# What is LoRA and QLoRA?
**LoRA (Low-Rank Adaptation)** is a method for fine-tuning large models by introducing low-rank matrices into pre-trained weights, significantly reducing the number of trainable parameters while maintaining performance.

**QLoRA (Quantized LoRA)** builds on this by incorporating quantization, which uses lower-precision data types to further reduce memory usage and computational requirements.

While both techniques aim to make fine-tuning more efficient, QLoRA is particularly advantageous for resource-constrained environments, as it enables fine-tuning on smaller hardware without sacrificing model accuracy.

## 📌 Why Fine-Tune a Model for Customer Support?
Large language models (LLMs) are powerful, but fine-tuning them on domain-specific data makes them more accurate, efficient, and cost-effective.

In this project, we:

✅ Use 4-bit quantization to **reduce memory usage**, making it possible to fine-tune large models on limited hardware.

✅ Implement LoRA to efficiently adapt the model to new data without retraining all parameters.

✅ Fine-tune the model on a customer support FAQs dataset **to enhance its ability** to respond to common user queries.

# 📂 Setup

In [ ]:
!pip install -qU transformers[torch] datasets

In [ ]:
!pip install -qU bitsandbytes trl peft accelerate

## Import libraries

In [2]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, TrainingArguments
import torch
from datasets import load_dataset
from peft import get_peft_model, LoraConfig
from trl import SFTTrainer

ModuleNotFoundError: No module named 'datasets'

## 𝌭 Model preparation

### Model config

In [ ]:
class CFG:
    model = "Qwen/Qwen2.5-1.5B-Instruct"

### Quantization config

Quantization reduces the model’s memory footprint, allowing it to run efficiently on standard hardware. Here’s how we configure it:

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,  # Load model in 4-bit precision
    bnb_4bit_quant_type='nf4',  # NormalFloat4 quantization for better accuracy
    bnb_4bit_compute_dtype=torch.float16,  # Compute using float16
    bnb_4bit_use_double_quant=True  # Enable double quantization for numerical stability
)

device_map = {"": torch.cuda.current_device()} if torch.cuda.is_available() else None

### Model creation

In [ ]:
model = AutoModelForCausalLM.from_pretrained(
    CFG.model,
    device_map=device_map,
    quantization_config=bnb_config,
)

# 📖 Tokenization and Dataset Preparation

## Tokenization

We need to tokenize the dataset correctly before training:

In [8]:
tokenizer = AutoTokenizer.from_pretrained(CFG.model, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token  # Ensures padded tokens don’t affect training
tokenizer.padding_side = "right"  # Ensures alignment with causal models

# set pad_token_id equal to the eos_token_id if not set
if tokenizer.pad_token_id is None:
  tokenizer.pad_token_id = tokenizer.eos_token_id

# Set reasonable default for models without max length
if tokenizer.model_max_length > 100_000:
  tokenizer.model_max_length = 2048

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

Many models don’t have a default pad token, so we set it to the EOS token to prevent issues when batching sequences.

Setting the padding to the right allows causal models to predict the next token correctly without looking ahead.

## 📌 Load and Preprocess the Dataset

In [3]:
dataset = load_dataset("Victorano/customer-support-1k", split="train")
dataset[0]

NameError: name 'load_dataset' is not defined

In [10]:
dataset = dataset.remove_columns(['flags', 'category','intent','text'])
dataset[0]

{'instruction': 'where do i enter a different shipping address',
 'response': 'Ah, I understand that you\'re looking to enter a different shipping address. Allow me to guide you through the process:\n\n1. Log in to your account on our website.\n2. Navigate to the "My Account" or "Profile" section, which can usually be found in the top right corner of the page.\n3. Look for the "Shipping Addresses" or similar option. It may also be labeled as "Manage Addresses" or "Delivery Information."\n4. Click on that option to access your saved addresses.\n5. To enter a different shipping address, you\'ll most likely have the choice to either "Edit" an existing address or "Add a New Address."\n6. If you choose to edit, find the address you want to update and click on the "Edit" or "Modify" button. Make your changes and then save.\n7. If you want to add a completely new address, click on the "Add a New Address" or a similar option. Fill in the necessary details and save.\n\nRemember to double-check 

In [11]:
dataset = dataset.train_test_split(test_size=0.2)

We structure each training example as a chat format:

In [12]:
instruction = """You are a helpful customer support bot..."""
def template(row):
    row_json = [
        {"role": "system", "content": instruction},
        {"role": "user", "content": row["instruction"]},
        {"role": "assistant", "content": row["response"]}
    ]
    row["text"] = tokenizer.apply_chat_template(row_json, tokenize=False)
    return row

dataset = dataset.map(template, num_proc=4)

Map (num_proc=4):   0%|          | 0/800 [00:00<?, ? examples/s]

Map (num_proc=4):   0%|          | 0/200 [00:00<?, ? examples/s]

> By formatting our data as system-user-assistant exchanges, we provide better conversational flow for training.

# 🔄 Fine-Tuning with QLoRA

**LoRA (Low-Rank Adaptation)** allows us to fine-tune LLMs efficiently by adding small trainable layers instead of modifying all parameters.

In [ ]:
lora_config = LoraConfig(
    r=4,  # Rank for low-rank matrices
    lora_alpha=8,  # Scaling factor
    lora_dropout=0.2,  # Regularization
    task_type="CAUSAL_LM",  # Language modeling task
    bias="none",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
)

In [14]:
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 544,768 || all params: 1,544,259,072 || trainable%: 0.0353


Now, setup training arguments:

In [ ]:
output_dir = 'data/Qwen2.5-1.5B-Instruct-lora'

training_arguments = TrainingArguments(
    output_dir=output_dir,  # Directory where model checkpoints and logs will be saved
    num_train_epochs=3,  # Number of training epochs (full passes through the dataset)
    per_device_train_batch_size=1,  # Batch size for training per device (GPU/CPU)
    per_device_eval_batch_size=1,  # Batch size for evaluation per device
    warmup_steps=5,  # Number of warmup steps for learning rate scheduler
    learning_rate=2e-4,  # Initial learning rate for optimizer
    fp16=True,  # Enable mixed precision training (float16) for performance optimization
    report_to="none",  # Disable reporting to tracking tools like TensorBoard or Weights & Biases
)

Start training using SFTTrainer:

In [16]:
trainer = SFTTrainer(
    model=model,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    tokenizer=tokenizer,
    args=training_arguments,
    peft_config=lora_config,
)

trainer.train()

<ipython-input-16-a9acd5980ba2>:1: FutureWarning: `tokenizer` is deprecated and removed starting from version 0.16.0 for `SFTTrainer.__init__`. Use `processing_class` instead.
  trainer = SFTTrainer(


Converting train dataset to ChatML:   0%|          | 0/800 [00:00<?, ? examples/s]

Applying chat template to train dataset:   0%|          | 0/800 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/800 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/800 [00:00<?, ? examples/s]

...

Tokenizing eval dataset:   0%|          | 0/200 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/200 [00:00<?, ? examples/s]

No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.


Step,Training Loss
500,1.271900
1000,1.011200
1500,0.939300
2000,0.887000


TrainOutput(global_step=2400, training_loss=1.0018185551961263, metrics={'train_runtime': 669.15, 'train_samples_per_second': 3.587, 'train_steps_per_second': 3.587, 'total_flos': 3031558764079104.0, 'train_loss': 1.0018185551961263})

# 📢 Generating Responses

Test the fine-tuned model:

`eos_token_id=tokenizer.eos_token_id` - Many language models use a special `"end of sequence" (EOS)` token to know when to stop generating.

In [23]:
def generate(input_prompt, model, tokenizer):
    messages = [{"role": "system", "content": instruction},
                {"role": "user", "content": input_prompt}]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors='pt', padding=True, truncation=True).to("cuda")
    outputs = model.generate(**inputs, max_new_tokens=2048, eos_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

In [ ]:
response = generate("Where can I see payment options?", model, tokenizer)
print(response)

# 💾 Save and Reload the Model

Save the trained model:

In [ ]:
output_dir = 'data/Qwen2.5-1.5B-Instruct-lora_save_model'
model.save_model(output_dir)

In [20]:
model.save_pretrained("./customer-faq-Qwen2.5-1.5B-Instruct")

# ␡ Delete model from memory

In [35]:
import gc

def memory_clean_up(model, tokenizer):
  del model
  del tokenizer
  gc.collect()
  if torch.cuda.is_available():
      torch.cuda.empty_cache()

In [ ]:
memory_clean_up()

# ⟳ Load model from path

In [29]:
# Try float16 first
try:
    model = AutoModelForCausalLM.from_pretrained(output_dir, load_in_4bit=True, torch_dtype=torch.float16, device_map="auto")
    print("Loaded with torch.float16")
except:
    print("Failed with float16, trying bfloat16...")
    model = AutoModelForCausalLM.from_pretrained(output_dir, load_in_4bit=True, torch_dtype=torch.bfloat16, device_map="auto")

## Load the tokenizer

In [26]:
tokenizer = AutoTokenizer.from_pretrained(output_dir)

## Load model

**General Guide Based on Hardware:**

  | GPU Series        | Recommended `torch_dtype` |
  |-------------------|--------------------------|
  | NVIDIA A100, H100 | `torch.bfloat16`         |
  | NVIDIA RTX 30XX, 40XX | `torch.bfloat16`   |
  | NVIDIA RTX 20XX, T4 | `torch.float16`        |
  | Older GPUs (GTX 10XX, P100, etc.) | `torch.float32` (No mixed precision support) |
  | CPU Only | `torch.float32` |


In [31]:
# Ensure model is in evaluation mode
model.eval()

Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 1536)
    (layers): ModuleList(
      (0-27): 28 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear(in_features=1536, out_features=1536, bias=True)
          (k_proj): Linear(in_features=1536, out_features=256, bias=True)
          (v_proj): Linear(in_features=1536, out_features=256, bias=True)
          (o_proj): Linear(in_features=1536, out_features=1536, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=1536, out_features=8960, bias=False)
          (up_proj): Linear(in_features=1536, out_features=8960, bias=False)
          (down_proj): Linear(in_features=8960, out_features=1536, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): Qwen2RMSNorm((1536,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((1536,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm((1536,), eps=1e-06)
    (rotary_emb): Qw

## Validate response

In [32]:
response = generate("Where can I see payment options?", model, tokenizer)
print(response)

system
You are a helpful customer support bot...
user
Where can I see payment options?
assistant
I'm here to assist you in locating the available payment options for your purchase. To find this information, please visit our website and navigate to the "Checkout" page or select "Payment Methods" from the main menu. On that page, you will be able to view a list of all accepted payment methods and their respective details. If you encounter any difficulties or have further questions about accessing these options, feel free to ask! We're here to ensure a smooth experience for you. How else can I help you today?


In [36]:
memory_clean_up(model, tokenizer)